# L06 · Real training and evaluation

## Goal

**Estimated time:** 50 min · **Path:** full

- preflight hardware
- read pinned presets
- interpret safe failures

### Current position: L05 → **L06** → L07

```text
Prompt/Data -> state source -> ... -> L06 -> ... -> fair evaluation
```

Alt text: The course map highlights L06 between its prerequisite and next lesson; every method remains connected to the same evaluation stage.

## Setup

In [1]:
LESSON_ID = "L06"
from pathlib import Path
import sys
import torch

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = Path.cwd().parents[1]
sys.path.insert(0, str(repo_root / "src"))

import opd_study
from opd_study.device import resolve_device
from opd_study.utils import seed_everything

seed_everything(42)
device_report = resolve_device("cpu")
print({"lesson": LESSON_ID, "opd_study": opd_study.__version__,
       "torch": torch.__version__, "device": device_report.selected,
       "profile": "toy", "network": "not required"})

{'lesson': 'L06', 'opd_study': '0.1.0.dev0', 'torch': '2.13.0', 'device': 'cpu', 'profile': 'toy', 'network': 'not required'}


## Steps

### 1/3 · 8–12 min

The real-model path shows revisions, licenses, size, and device before download. This notebook defaults to an offline toy fallback and never fabricates network results.

Figure alt: labels and numbers remain readable without color.

### Core mechanics

The real path begins with **preflight**, not training. Check dataset/model revisions, expected bytes, license consent, tokenizer/chat template, dtype, device, and fallback policy before download. The official test split is never trained on; validation is deterministically carved from train.

LoRA learns a small low-rank update `B·A` over frozen weights. QLoRA also quantizes base weights to 4-bit, reducing VRAM further, but the bitsandbytes/CUDA combination needs a real forward-backward-save-reload probe. This repository never silently falls back to full fine-tuning.

### Production implementation: why this design

Research configs carry `backend=research`, exact 40-character revisions, expected bytes, and consent flags. Preflight checks real imports, PyTorch version, device, and QLoRA capability—not merely package presence. The loader fails on vocabulary or chat-template mismatch.

Production code: [`preflight.py`](../../src/opd_study/research/preflight.py), [`hf_backend.py`](../../src/opd_study/research/hf_backend.py).

In [2]:
import inspect
from opd_study.research import research_preflight

objects_to_show = (research_preflight,)
for object_to_show in objects_to_show:
    source_lines = inspect.getsource(object_to_show).splitlines()
    print(f"\n# {object_to_show.__module__}.{object_to_show.__qualname__}")
    print("\n".join(source_lines[:80]))
    if len(source_lines) > 80:
        print(f"... {len(source_lines) - 80} more lines; open the linked source file")


# opd_study.research.preflight.research_preflight
def research_preflight(config: ExperimentConfig) -> ResearchPreflight:
    if config.backend != "research":
        raise ValueError("research preflight requires backend=research")
    required = ["transformers", "datasets", "accelerate", "peft"]
    if config.model.finetuning == "qlora":
        required.append("bitsandbytes")
    missing = tuple(name for name in required if importlib.util.find_spec(name) is None)
    blockers: list[str] = []
    if missing:
        blockers.append("missing optional packages: " + ", ".join(missing))
    package_versions: dict[str, str] = {}
    for distribution in ("torch", *required):
        try:
            package_versions[distribution] = version(distribution)
        except PackageNotFoundError:
            continue
    if Version(package_versions["torch"]) < Version("2.2"):
        blockers.append(
            f"research backend requires torch>=2.2; found {package_versions['torch']}"
        )
 

### Alternatives and trade-offs

Full fine-tuning is direct but memory-heavy. LoRA is broadly stable; QLoRA saves VRAM but adds CUDA/bitsandbytes constraints. On macOS, do not substitute full FT for failed QLoRA—choose LoRA/CPU toy or validated CUDA.

### 2/3 · Run and observe

Predict before running: which invariant should you inspect first in L06's output? Write one sentence, then run.

In [3]:
from opd_study.config import load_config
from opd_study.device import require_qlora

toy = load_config(repo_root / "configs/toy/default.yaml")
laptop = load_config(repo_root / "configs/laptop/gsm8k_lora.yaml")
qlora = load_config(repo_root / "configs/laptop/gsm8k_qlora.yaml")
print("toy:", toy.profile, toy.backend, toy.data.id)
print("laptop pins:", laptop.model.student, laptop.model.student_revision)
print("CUDA preset:", qlora.model.finetuning, qlora.training.device, qlora.training.precision)
print("download estimate is documented before any network call:", laptop.data.expected_download_bytes)

toy: toy mini tiny_arithmetic
laptop pins: Qwen/Qwen3-0.6B c1899de289a04d12100db370d81485cdf75e47ca
CUDA preset: qlora cuda float16
download estimate is documented before any network call: 2725633


In [4]:
try:
    require_qlora(device_report)
except RuntimeError as error:
    print("safe QLoRA block:", error)

RUN_OPTIONAL_NETWORK = False
print("Qwen/GSM8K smoke enabled:", RUN_OPTIONAL_NETWORK)
print("Exact command: opd-study research-train --config configs/laptop/gsm8k_lora.yaml --smoke --accept-dataset-license --accept-model-license")

safe QLoRA block: QLoRA requires a validated NVIDIA CUDA + bitsandbytes environment in this project; macOS/MPS and CPU never fall back to full fine-tuning
Qwen/GSM8K smoke enabled: False
Exact command: opd-study research-train --config configs/laptop/gsm8k_lora.yaml --smoke --accept-dataset-license --accept-model-license


## Checks

In [5]:
assert len(laptop.model.student_revision) == 40
assert laptop.model.trust_remote_code is False
assert qlora.model.finetuning == "qlora" and qlora.training.device == "cuda"
assert RUN_OPTIONAL_NETWORK is False
print("check passed: pinned assets, no remote code, unsupported QLoRA does not fall back")

check passed: pinned assets, no remote code, unsupported QLoRA does not fall back


**Exercise (8 min):** write a failure report for `device=mps` plus `finetuning=qlora`. Do not suggest silent full-FT fallback.

<details><summary>Check</summary>Name bitsandbytes/CUDA constraints, expected download and consent, plus LoRA/CPU-toy alternatives.</details>

## My recurring mistakes

### M1 — Pinning only a model name

- Wrong: leave revision at latest.
- Why: code, config, and weights can move.
- Fix: record a 40-char revision, license, bytes, and checksums.
- Related check: `test_all_checked_in_presets_parse`

### M2 — Hiding QLoRA failure with full FT

- Wrong: silently switch to a larger-memory path when bitsandbytes fails.
- Why: this hides OOM risk and changes experiment meaning.
- Fix: block explicitly and offer LoRA or validated CUDA.
- Related check: `test_qlora_preset_is_explicit_cuda_and_opt_in`

## 60-second summary

1. preflight hardware
2. read pinned presets
3. interpret safe failures

## Next Steps

Before the next notebook, rerun the assertions and record one prediction you revised.

### Sources

- [`gkd`](https://arxiv.org/abs/2306.13649v3) · `2306.13649v3` · license `CC-BY-4.0` · [audited manifest](../../docs/sources.yml)
- [`openai/gsm8k`](https://huggingface.co/datasets/openai/gsm8k) · `740312add88f781978c0658806c59bc2815b9866` · license `MIT` · [audited manifest](../../docs/sources.yml)
- [`Qwen/Qwen3-0.6B`](https://huggingface.co/Qwen/Qwen3-0.6B) · `c1899de289a04d12100db370d81485cdf75e47ca` · license `Apache-2.0` · [audited manifest](../../docs/sources.yml)
- [`Qwen/Qwen3-1.7B`](https://huggingface.co/Qwen/Qwen3-1.7B) · `70d244cc86ccca08cf5af4e1e306ecf908b1ad5e` · license `Apache-2.0` · [audited manifest](../../docs/sources.yml)